In [2]:
import pandas as pd

# -----------------------
# Load Excel file
# -----------------------
file_path = "/mnt/f/Solar-Review-Ppaer/Monthwise-Data-Sheets/Data_clean/18. Sept-2025-filtered.xlsx"
df = pd.read_excel(file_path)

# -----------------------
# Clean column names (important for Excel files)
# -----------------------
df.columns = df.columns.str.strip()

# -----------------------
# Step 1: Convert Time & filter 6 AM–6 PM
# -----------------------
df["Time_dt"] = pd.to_datetime(
    df["Time"].astype(str),
    format="%H.%M.%S",
    errors="coerce"
).dt.time


# -----------------------
# Step 2: Create DateTime column
# -----------------------
df["DateTime"] = pd.to_datetime(
    df["Year"].astype(str) + "-" +
    df["Month"].astype(str).str.zfill(2) + "-" +
    df["Date"].astype(str).str.zfill(2) + " " +
    df["Time"].astype(str).str.replace(".", ":", regex=False),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)
df = df[
    (df["Time_dt"] >= pd.to_datetime("06:00:00").time()) &
    (df["Time_dt"] <= pd.to_datetime("18:00:00").time())
]



In [3]:
import pandas as pd
import numpy as np

# Replace '*' with NaN
df.replace('*', np.nan, inplace=True)

# Drop rows where GHI or Temperature is NaN or zero
df = df.dropna(subset=['GHI', 'Temperature'])
df = df[(df['GHI'] != 0) & (df['Temperature'] != 0)]

# Reset index
df.reset_index(drop=True, inplace=True)

# Check again
print("Missing/NaN values per column after cleaning important columns:")
print(df.isna().sum())
print("\nZero values per numeric column after cleaning important columns:")
print((df.select_dtypes(include='number') == 0).sum())


Missing/NaN values per column after cleaning important columns:
Time           0
Date           0
Month          0
Year           0
Temperature    0
Humidity       0
Pressure       0
GHI            0
Time_dt        0
DateTime       0
dtype: int64

Zero values per numeric column after cleaning important columns:
Date           0
Month          0
Year           0
Temperature    0
Humidity       0
Pressure       0
GHI            0
dtype: int64


In [4]:
df_selected = df[['DateTime', 'GHI', 'Temperature']].copy()

df_selected["GHI"] = pd.to_numeric(df_selected["GHI"], errors="coerce")
df_selected["Temperature"] = pd.to_numeric(df_selected["Temperature"], errors="coerce")
df_selected["DateTime"] = pd.to_datetime(df_selected["DateTime"], errors="coerce")

df_selected = df_selected.dropna(subset=["DateTime"])
df_selected = df_selected.sort_values("DateTime")
df_selected = df_selected.set_index("DateTime")

# Time filter
df_selected = df_selected.between_time("06:00", "17:30")

# Bring DateTime back as column (optional but useful)
#df_selected = df_selected.reset_index()


In [7]:

# -----------------------
# Resampling
# -----------------------
# 15-minute
df_10min = (
    df_selected
    .resample('10T').mean()
    .reset_index()
    .dropna(subset=['GHI', 'Temperature'], how='all')
)


# -----------------------
# Save outputs
# -----------------------
df_10min.to_excel(
    "/mnt/f/Solar-Review-Ppaer/Monthwise-Data-Sheets/Data_clean/10-Minute-Average-Data/18.Sep-2025-filtered-10-min-Avg.xlsx",
    index=False
)

